# PRISM-FUSION v4 — Unified Multimodal Video Title Generation

**One model handles all video types** — text-rich, silent, or mixed.

| Modality | Feature | Shape | Carries |
|---|---|---|---|
| Visual | CLIP5 (5 temporal frames) | (5, 512) | Scene, objects, temporal change |
| Audio Events | CLAP | (512,) | Crash sounds, engine, music, speech |
| On-screen text | OCR-BERT | (768,) | Text overlays, chyrons, product labels |
| Speech | ASR-BERT | (768,) | Narration, dialogue, commentary |

**Quality weighting** scales each modality by how informative it is for that specific video — ASR dominates tutorials, CLIP+CLAP dominate silent events.

**Training:** Phase 0 (contrastive warmup) → Phase 1 (LoRA align) → Phase 2 (joint fine-tune)

## Cell 1 — Install Dependencies

In [1]:
!pip install -q transformers==4.40.0
!pip install -q peft>=0.10.0
!pip install -q bert-score rouge-score nltk
!pip install -q sentence-transformers
import subprocess; subprocess.run(['python','-c','import peft; print("peft",peft.__version__)'])
print('All installs done')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 98.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 136.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 111.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 116.8 MB/s eta 0:00:00
All inst

## Cell 2 — Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

Mounted at /content/drive
Drive mounted


## Cell 3 — Config

In [21]:
from pathlib import Path
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# Paths
BASE      = Path('/content/drive/MyDrive/videostory_prism')
FEAT_DIR  = BASE / 'features'
CLIP5_DIR = FEAT_DIR / 'clip5'
CLAP_DIR  = FEAT_DIR / 'clap'
BERT_DIR  = FEAT_DIR / 'bert'
OCR_DIR   = BASE / 'ocr_text'   # {vid}.txt
ASR_DIR   = BASE / 'audio_text' # {vid}.txt
GT_CSV    = BASE / 'final.csv'
CKPT_DIR  = BASE / 'checkpoints_v4'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Model dims
CLIP_DIM     = 512
CLAP_DIM     = 1024
BERT_DIM     = 768
EMBED_DIM    = 768
FUSION_HEADS = 8
BART_MODEL   = 'facebook/bart-base'
MAX_LABEL    = 20
MAX_NEW_TOK  = 20

# Training
BATCH_SIZE      = 16
ACCUM_STEPS     = 2     # effective batch = 32
PHASE0_EPOCHS   = 10
PHASE1_EPOCHS   = 20
PHASE2_EPOCHS   = 10

# Verify feature dirs
for name, d in [('clip5', CLIP5_DIR), ('clap', CLAP_DIR),
                ('bert',  BERT_DIR),  ('ocr', OCR_DIR), ('asr', ASR_DIR)]:
    n = len(list(d.glob('*'))) if d.exists() else 0
    print(f'  {name:6s}: {n} files  {"OK" if n > 0 else "<-- missing!"}')

Device: cuda
  clip5 : 4500 files  OK
  clap  : 4629 files  OK
  bert  : 9000 files  OK
  ocr   : 4500 files  OK
  asr   : 4500 files  OK


## Cell 4 — Quality Score Utilities

In [4]:
import re, math

def compute_quality(text: str) -> float:
    '''
    Soft quality score in [0, 1] for a text modality.
    0.0 = no usable text   1.0 = rich, clean speech or OCR
    Based on count of real English words (len >= 2, alpha-only).
    '''
    if not text or not text.strip():
        return 0.0
    words = [w for w in text.split() if re.match(r'^[A-Za-z]{2,}$', w)]
    return min(len(words) / 30.0, 1.0)


def load_text(path: Path) -> str:
    if path.exists():
        return path.read_text(encoding='utf-8', errors='ignore').strip()
    return ''


print('Quality utils ready')

Quality utils ready


## Cell 5 — Load Ground Truth & Build Dataset

In [17]:
import pandas as pd, numpy as np
from transformers import BartTokenizer
from tqdm.auto import tqdm

bart_tokenizer = BartTokenizer.from_pretrained(BART_MODEL)
gt_df = pd.read_csv(GT_CSV, encoding='latin-1')

# For each video, verify all 4 feature files exist and precompute quality scores
records = []
for _, row in tqdm(gt_df.iterrows(), total=len(gt_df), desc='Building dataset'):
    vid   = row['video_id']
    title = str(row['title'])

    c5_f  = CLIP5_DIR / f'{vid}.npy'
    cl_f  = CLAP_DIR  / f'{vid}.npy'
    ocr_f = BERT_DIR  / f'{vid}_ocr.npy'
    asr_f = BERT_DIR  / f'{vid}_asr.npy'
    if not all(f.exists() for f in [c5_f, cl_f, ocr_f, asr_f]):
        continue   # skip incomplete videos

    ocr_txt = load_text(OCR_DIR / f'{vid}.txt')
    asr_txt = load_text(ASR_DIR / f'{vid}.txt')

    records.append({
        'video_id': vid,
        'title':    title,
        'q_ocr':    compute_quality(ocr_txt),
        'q_asr':    compute_quality(asr_txt),
    })

df = pd.DataFrame(records).reset_index(drop=True)
print(f'Videos ready for training: {len(df)}')
print(f'  OCR available (q>0): {(df.q_ocr > 0).sum()} ({100*(df.q_ocr>0).mean():.1f}%)')
print(f'  ASR available (q>0): {(df.q_asr > 0).sum()} ({100*(df.q_asr>0).mean():.1f}%)')
print(f'  Both zero (silent):  {((df.q_ocr==0)&(df.q_asr==0)).sum()}')

Building dataset:   0%|          | 0/4127 [00:00<?, ?it/s]

Videos ready for training: 4127
  OCR available (q>0): 2131 (51.6%)
  ASR available (q>0): 2625 (63.6%)
  Both zero (silent):  792


## Cell 6 — PRISMDatasetV4

In [18]:
import torch, numpy as np
from torch.utils.data import Dataset, DataLoader, random_split

class PRISMDatasetV4(Dataset):
    '''
    Returns per-sample:
      clip5  (5, 512)  — 5-frame CLIP embeddings
      clap   (512,)   — CLAP audio event embedding
      ocr    (768,)   — OCR BERT CLS
      asr    (768,)   — ASR BERT CLS
      q_vis  scalar   — always 1.0 (CLIP always extracts)
      q_audio scalar  — CLAP L2 norm / 5, capped at 1
      q_ocr  scalar   — quality of OCR text (0=none, 1=rich)
      q_asr  scalar   — quality of ASR text (0=none, 1=rich)
      labels (L,)     — tokenized title (-100 on padding)
      title  str      — raw title (for Phase 0 BERT encoding)
    '''
    def __init__(self, df, tokenizer, max_label_len=MAX_LABEL):
        self.df  = df.reset_index(drop=True)
        self.tok = tokenizer
        self.L   = max_label_len

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        vid  = row['video_id']

        clip5 = torch.from_numpy(np.load(CLIP5_DIR / f'{vid}.npy')).float()
        clap  = torch.from_numpy(np.load(CLAP_DIR  / f'{vid}.npy')).float()
        ocr   = torch.from_numpy(np.load(BERT_DIR  / f'{vid}_ocr.npy')).float()
        asr   = torch.from_numpy(np.load(BERT_DIR  / f'{vid}_asr.npy')).float()

        q_vis   = torch.tensor(1.0, dtype=torch.float)
        q_audio = torch.tensor(min(clap.norm().item(), 1.0), dtype=torch.float)
        q_ocr   = torch.tensor(float(row['q_ocr']), dtype=torch.float)
        q_asr   = torch.tensor(float(row['q_asr']), dtype=torch.float)

        enc = self.tok(
            row['title'], max_length=self.L, truncation=True,
            padding='max_length', return_tensors='pt'
        )
        labels = enc['input_ids'].squeeze(0).clone()
        labels[labels == self.tok.pad_token_id] = -100

        return (clip5, clap, ocr, asr,
                q_vis, q_audio, q_ocr, q_asr,
                labels, row['title'])


def collate_v4(batch):
    c5, cl, oc, ar, qv, qa, qo, qs, lbl, titles = zip(*batch)
    return (
        torch.stack(c5), torch.stack(cl), torch.stack(oc), torch.stack(ar),
        torch.stack(qv), torch.stack(qa), torch.stack(qo), torch.stack(qs),
        torch.stack(lbl), list(titles)
    )


# Train/Val/Test split 85/10/5
n     = len(df)
n_val = max(1, int(0.10 * n))
n_tst = max(1, int(0.05 * n))
n_trn = n - n_val - n_tst

full_ds = PRISMDatasetV4(df, bart_tokenizer)
trn_ds, val_ds, tst_ds = random_split(
    full_ds, [n_trn, n_val, n_tst],
    generator=torch.Generator().manual_seed(42)
)
trn_dl = DataLoader(trn_ds, batch_size=BATCH_SIZE, shuffle=True,
                    collate_fn=collate_v4, num_workers=2, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                    collate_fn=collate_v4, num_workers=2, pin_memory=True)
print(f'Train: {n_trn} | Val: {n_val} | Test: {n_tst} | Batches/ep: {len(trn_dl)}')

Train: 3509 | Val: 412 | Test: 206 | Batches/ep: 220


## Cell 7 — PRISMFusionEncoder + TemporalAttention

In [22]:
import torch.nn as nn
from transformers.modeling_outputs import BaseModelOutput

# ── Temporal Attention ────────────────────────────────────────────────────
class TemporalAttention(nn.Module):
    '''
    Aggregates N CLIP frame embeddings into a single vector via
    cross-attention pooling with a learnable query token.
    Input:  (B, N, dim) -> Output: (B, dim)
    '''
    def __init__(self, dim=512, n_frames=5, n_heads=4):
        super().__init__()
        self.pos   = nn.Embedding(n_frames, dim)
        self.attn  = nn.MultiheadAttention(dim, n_heads, batch_first=True, dropout=0.1)
        self.query = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.norm  = nn.LayerNorm(dim)

    def forward(self, x):
        B, N, D = x.shape
        pos = self.pos(torch.arange(N, device=x.device))
        x   = x + pos.unsqueeze(0)
        q   = self.query.expand(B, -1, -1)
        out, _ = self.attn(q, x, x)
        return self.norm(out.squeeze(1))   # (B, dim)


# ── Fusion Encoder ────────────────────────────────────────────────────────
class PRISMFusionEncoder(nn.Module):
    '''
    4-modality fusion encoder.  Produces (B, 4, 768) fused token sequence
    ready to be consumed as encoder_outputs by BART decoder.

    Quality weighting:
      each projected token is interpolated between:
        quality=1 -> full projected embedding
        quality=0 -> learned empty embedding
      allowing the model to express "no audio", "no OCR", "no speech"
      in a learnable, continuous way.
    '''
    def __init__(self):
        super().__init__()

        # Temporal aggregation for 5 CLIP frames
        self.temporal_attn = TemporalAttention(dim=CLIP_DIM, n_frames=5, n_heads=4)

        # Modality projections -> EMBED_DIM
        self.vis_proj   = nn.Linear(CLIP_DIM, EMBED_DIM)
        self.event_proj = nn.Linear(CLAP_DIM, EMBED_DIM)
        self.ocr_proj   = nn.Linear(BERT_DIM, EMBED_DIM)
        self.asr_proj   = nn.Linear(BERT_DIM, EMBED_DIM)

        # Learned "empty" tokens for zero-quality modalities
        self.empty_event = nn.Parameter(torch.randn(EMBED_DIM) * 0.02)
        self.empty_ocr   = nn.Parameter(torch.randn(EMBED_DIM) * 0.02)
        self.empty_asr   = nn.Parameter(torch.randn(EMBED_DIM) * 0.02)

        # Positional embeddings for 4 modality slots
        self.pos_embed = nn.Embedding(4, EMBED_DIM)

        # 1-layer cross-modal transformer
        enc_layer = nn.TransformerEncoderLayer(
            d_model=EMBED_DIM, nhead=FUSION_HEADS,
            dim_feedforward=EMBED_DIM * 4, dropout=0.1,
            batch_first=True, norm_first=True
        )
        self.fusion = nn.TransformerEncoder(enc_layer, num_layers=1)
        self.norm   = nn.LayerNorm(EMBED_DIM)

    def _mix(self, projected, empty_emb, quality):
        '''
        Soft interpolation: quality * projected + (1-quality) * empty
        quality: (B,) in [0,1]
        '''
        B  = projected.shape[0]
        q  = quality.view(B, 1).to(projected.device)
        em = empty_emb.unsqueeze(0).expand(B, -1)
        return q * projected + (1 - q) * em

    def forward(self, clip5, clap, ocr_emb, asr_emb,
                q_vis, q_audio, q_ocr, q_asr):
        '''
        clip5   : (B, 5, 512)
        clap    : (B, 512)
        ocr_emb : (B, 768)
        asr_emb : (B, 768)
        q_*     : (B,)  quality scores in [0,1]
        returns : (B, 4, 768)  fused cross-modal tokens
        '''
        B = clip5.shape[0]

        # Visual — temporal aggregation then project
        vis_temp = self.temporal_attn(clip5)    # (B, 512)
        v_tok    = self.vis_proj(vis_temp)      # (B, 768)  quality is always 1.0

        # Audio events — quality blend
        e_proj = self.event_proj(clap)          # (B, 768)
        e_tok  = self._mix(e_proj, self.empty_event, q_audio)

        # OCR — quality blend
        o_proj = self.ocr_proj(ocr_emb)         # (B, 768)
        o_tok  = self._mix(o_proj, self.empty_ocr, q_ocr)

        # ASR — quality blend
        a_proj = self.asr_proj(asr_emb)         # (B, 768)
        a_tok  = self._mix(a_proj, self.empty_asr, q_asr)

        # Stack -> fuse -> normalize
        tokens = torch.stack([v_tok, e_tok, o_tok, a_tok], dim=1)  # (B,4,768)
        pos    = self.pos_embed(torch.arange(4, device=tokens.device))
        tokens = tokens + pos.unsqueeze(0)
        fused  = self.fusion(tokens)             # (B,4,768)
        return self.norm(fused)


# Quick shape test
with torch.no_grad():
    _enc = PRISMFusionEncoder()
    _c5  = torch.randn(4, 5, CLIP_DIM)
    _cl  = torch.randn(4, CLAP_DIM)
    _oc  = torch.randn(4, BERT_DIM);  _oc[0].zero_()
    _ar  = torch.randn(4, BERT_DIM)
    _qv  = torch.ones(4)
    _qa  = torch.tensor([0.8, 0.0, 0.5, 1.0])
    _qo  = torch.tensor([0.0, 0.7, 0.0, 0.9])
    _qs  = torch.tensor([0.9, 0.0, 0.6, 0.8])
    _out = _enc(_c5, _cl, _oc, _ar, _qv, _qa, _qo, _qs)
    print(f'PRISMFusionEncoder output: {_out.shape}  <- expect (4, 4, 768)')
    p = sum(x.numel() for x in _enc.parameters())
    print(f'FusionEncoder params: {p/1e6:.2f}M')
    del _enc, _c5, _cl, _oc, _ar, _out

PRISMFusionEncoder output: torch.Size([4, 4, 768])  <- expect (4, 4, 768)
FusionEncoder params: 10.51M


/tmp/ipykernel_9312/2735647944.py:66: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.fusion = nn.TransformerEncoder(enc_layer, num_layers=1)


## Cell 8 — Phase 0: Contrastive Warm-Up

**Goal:** Push the fusion encoder output toward the semantic space of GT titles.

**How it works:**
```
4 modalities → FusionEncoder → mean-pool → (B, 768)
GT titles    → frozen BERT  →  CLS       → (B, 768)
Loss = 1 - cosine_similarity(fused_pool, title_bert).mean()
```
**Why this matters:** Gives the fusion encoder a semantic 'north star' before
BART is involved. Without this, BART sees meaningless random projections and collapses.

> **Time:** ~20-30 min for 10 epochs on T4

In [ ]:
from transformers import BertTokenizer, BertModel
import torch.nn.functional as F
from torch.optim import AdamW
from torch.amp import GradScaler, autocast
import json

# Load frozen BERT for title encoding
print('Loading BERT for title encoding...')
title_bert_tok = BertTokenizer.from_pretrained('bert-base-uncased')
title_bert     = BertModel.from_pretrained('bert-base-uncased').to(DEVICE)
title_bert.eval()
for p in title_bert.parameters():
    p.requires_grad = False
print('BERT ready (frozen)')

@torch.no_grad()
def encode_titles_bert(titles):
    '''Batch encode title strings -> (B, 768) CLS embeddings.'''
    enc  = title_bert_tok(
        list(titles), max_length=32, truncation=True,
        padding='max_length', return_tensors='pt'
    )
    ids  = enc['input_ids'].to(DEVICE)
    mask = enc['attention_mask'].to(DEVICE)
    out  = title_bert(input_ids=ids, attention_mask=mask)
    return out.last_hidden_state[:, 0, :]   # CLS token


# ── Train Phase 0 ─────────────────────────────────────────────────────────
fusion_enc = PRISMFusionEncoder().to(DEVICE)
optimizer  = AdamW(fusion_enc.parameters(), lr=3e-4, weight_decay=0.01)
scaler     = GradScaler('cuda')

p0_history = {'train': [], 'val': []}
best_p0    = float('inf')

print(f'Starting Phase 0 — {PHASE0_EPOCHS} epochs')

for epoch in range(1, PHASE0_EPOCHS + 1):

    # Train
    fusion_enc.train()
    trn_loss, steps = 0.0, 0
    optimizer.zero_grad()

    for step, (c5, cl, oc, ar, qv, qa, qo, qs, lbl, titles) in enumerate(trn_dl):
        c5 = c5.to(DEVICE); cl = cl.to(DEVICE)
        oc = oc.to(DEVICE); ar = ar.to(DEVICE)
        qv = qv.to(DEVICE); qa = qa.to(DEVICE)
        qo = qo.to(DEVICE); qs = qs.to(DEVICE)

        with autocast('cuda'):
            fused      = fusion_enc(c5, cl, oc, ar, qv, qa, qo, qs)  # (B,4,768)
            fused_pool = fused.mean(dim=1)                             # (B,768)
            title_emb  = encode_titles_bert(titles)                    # (B,768)
            # Normalize both for stable cosine
            fp_norm    = F.normalize(fused_pool, dim=-1)
            te_norm    = F.normalize(title_emb,  dim=-1)
            cos_loss   = (1 - (fp_norm * te_norm).sum(dim=-1).mean()) / ACCUM_STEPS

        scaler.scale(cos_loss).backward()
        if (step + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(fusion_enc.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad()

        trn_loss += cos_loss.item() * ACCUM_STEPS; steps += 1

    avg_trn = trn_loss / steps

    # Val
    fusion_enc.eval()
    val_loss, vsteps = 0.0, 0
    with torch.no_grad():
        for c5, cl, oc, ar, qv, qa, qo, qs, lbl, titles in val_dl:
            c5 = c5.to(DEVICE); cl = cl.to(DEVICE)
            oc = oc.to(DEVICE); ar = ar.to(DEVICE)
            qv = qv.to(DEVICE); qa = qa.to(DEVICE)
            qo = qo.to(DEVICE); qs = qs.to(DEVICE)
            with autocast('cuda'):
                fused     = fusion_enc(c5, cl, oc, ar, qv, qa, qo, qs)
                pool      = F.normalize(fused.mean(dim=1), dim=-1)
                te        = F.normalize(encode_titles_bert(titles), dim=-1)
                val_loss += (1 - (pool * te).sum(dim=-1).mean()).item()
            vsteps += 1

    avg_val = val_loss / vsteps
    p0_history['train'].append(avg_trn)
    p0_history['val'].append(avg_val)

    print(f'[P0] Ep {epoch:02d}/{PHASE0_EPOCHS} | '
          f'train={avg_trn:.4f} | val={avg_val:.4f}')

    if avg_val < best_p0:
        best_p0 = avg_val
        torch.save({'epoch': epoch, 'model_state': fusion_enc.state_dict(),
                    'val_loss': avg_val},
                   CKPT_DIR / 'phase0_best.pt')
        print(f'  Saved Phase 0 checkpoint (val={avg_val:.4f})')

(CKPT_DIR / 'history_phase0.json').write_text(json.dumps(p0_history))
print(f'Phase 0 done. Best val cosine loss: {best_p0:.4f}')
print('Ideal range: 0.10 - 0.30  (lower = fusion closer to title space)')
torch.cuda.empty_cache()

# Sanity: avg cosine similarity
print(f'Avg cosine SIMILARITY (1 - loss): {1 - best_p0:.4f}')
print('  > 0.70 → Great alignment')
print('  0.50-0.70 → Good enough to proceed')
print('  < 0.50 → Run more epochs before Phase 1')

Loading BERT for title encoding...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT ready (frozen)
Starting Phase 0 — 10 epochs


/tmp/ipykernel_9312/2735647944.py:66: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.fusion = nn.TransformerEncoder(enc_layer, num_layers=1)


## Cell 9 — Phase 0 Diagnostics

In [ ]:
import json, matplotlib.pyplot as plt

h    = json.loads((CKPT_DIR / 'history_phase0.json').read_text())
eps  = list(range(1, len(h['train']) + 1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(eps, h['train'], label='train loss'); axes[0].plot(eps, h['val'], label='val loss')
axes[0].set_title('Phase 0 — Cosine Loss (lower=better alignment)')
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(eps, [1-v for v in h['train']], label='train sim')
axes[1].plot(eps, [1-v for v in h['val']],   label='val sim')
axes[1].set_title('Phase 0 — Cosine Similarity (higher=better)')
axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CKPT_DIR / 'phase0_loss.png', dpi=150)
plt.show()

## Cell 10 — PRISMFusionV4: Full Model (Fusion + BART-LoRA)

Loads the Phase 0 fusion encoder weights and attaches BART-base with LoRA adapters.

In [ ]:
from transformers import BartForConditionalGeneration
from peft import get_peft_model, LoraConfig, TaskType

class PRISMFusionV4(nn.Module):
    '''
    Full PRISM-FUSION v4 model.
    Fusion encoder: ~10.7M params (from Phase 0)
    BART decoder + LoRA rank=8: ~139M frozen + ~300K trainable
    '''
    def __init__(self, encoder_ckpt=None):
        super().__init__()

        # Fusion encoder — load Phase 0 weights if provided
        self.encoder = PRISMFusionEncoder()
        if encoder_ckpt is not None:
            ckpt = torch.load(encoder_ckpt, map_location='cpu')
            self.encoder.load_state_dict(ckpt['model_state'])
            print(f'  Loaded fusion encoder from Phase 0 (ep={ckpt["epoch"]})')

        # BART-base with LoRA adapters
        bart_base  = BartForConditionalGeneration.from_pretrained(BART_MODEL)
        lora_cfg   = LoraConfig(
            task_type     = TaskType.SEQ_2_SEQ_LM,
            r             = 8,
            lora_alpha    = 16,
            # Target BOTH self-attn and cross-attn in the decoder
            target_modules= ['q_proj', 'v_proj'],
            lora_dropout  = 0.1,
            bias          = 'none',
        )
        self.bart = get_peft_model(bart_base, lora_cfg)

    def _encode(self, clip5, clap, ocr, asr, qv, qa, qo, qs):
        fused = self.encoder(clip5, clap, ocr, asr, qv, qa, qo, qs)   # (B,4,768)
        attn  = torch.ones(fused.shape[:2], device=fused.device)
        return BaseModelOutput(last_hidden_state=fused), attn

    def forward(self, clip5, clap, ocr, asr, qv, qa, qo, qs, labels=None):
        enc_out, attn = self._encode(clip5, clap, ocr, asr, qv, qa, qo, qs)
        return self.bart(encoder_outputs=enc_out, attention_mask=attn, labels=labels)

    @torch.no_grad()
    def generate_title(self, clip5, clap, ocr, asr, qv, qa, qo, qs, tokenizer):
        self.eval()
        enc_out, attn = self._encode(clip5, clap, ocr, asr, qv, qa, qo, qs)
        ids = self.bart.generate(
            encoder_outputs  = enc_out,
            attention_mask   = attn,
            max_new_tokens   = MAX_NEW_TOK,
            min_length       = 4,
            num_beams        = 4,
            no_repeat_ngram_size = 2,
            early_stopping   = False,
            length_penalty   = 1.2,
        )
        return tokenizer.decode(ids[0], skip_special_tokens=True)


# Instantiate — loads Phase 0 fusion weights
model = PRISMFusionV4(encoder_ckpt=CKPT_DIR / 'phase0_best.pt').to(DEVICE)

# Parameter report
enc_p  = sum(p.numel() for p in model.encoder.parameters())
all_p  = sum(p.numel() for p in model.parameters())
lora_p = sum(p.numel() for p in model.bart.parameters() if p.requires_grad)
print(f'Fusion encoder params : {enc_p/1e6:.2f}M')
print(f'LoRA trainable params : {lora_p/1e3:.0f}K')
print(f'Total params          : {all_p/1e6:.1f}M')

## Cell 11 — Phase 1: LoRA Alignment (20 epochs)

**Fusion encoder is FROZEN.** Only LoRA adapters (~300K params) train.
BART decoder learns to decode multimodal tokens from a stable frozen encoder.

> **Time:** ~45-55 min on T4

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import GradScaler, autocast
import json

# Freeze fusion encoder
for p in model.encoder.parameters():
    p.requires_grad = False
trainable_p1 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Phase 1 trainable: {trainable_p1/1e3:.0f}K  (LoRA only)')

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=5e-4, weight_decay=0.01
)
scheduler = CosineAnnealingLR(optimizer, T_max=PHASE1_EPOCHS, eta_min=1e-5)
scaler    = GradScaler('cuda')

p1_history = {'train': [], 'val': []}
best_p1    = float('inf')

print(f'Starting Phase 1 — {PHASE1_EPOCHS} epochs')

for epoch in range(1, PHASE1_EPOCHS + 1):

    model.train()
    trn_loss, steps = 0.0, 0
    optimizer.zero_grad()

    for step, (c5, cl, oc, ar, qv, qa, qo, qs, lbl, _) in enumerate(trn_dl):
        c5  = c5.to(DEVICE);  cl  = cl.to(DEVICE)
        oc  = oc.to(DEVICE);  ar  = ar.to(DEVICE)
        qv  = qv.to(DEVICE);  qa  = qa.to(DEVICE)
        qo  = qo.to(DEVICE);  qs  = qs.to(DEVICE)
        lbl = lbl.to(DEVICE)

        with autocast('cuda'):
            out  = model(c5, cl, oc, ar, qv, qa, qo, qs, labels=lbl)
            loss = out.loss / ACCUM_STEPS

        scaler.scale(loss).backward()
        if (step + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad()

        trn_loss += out.loss.item(); steps += 1

    avg_trn = trn_loss / steps

    model.eval()
    val_loss, vsteps = 0.0, 0
    with torch.no_grad():
        for c5, cl, oc, ar, qv, qa, qo, qs, lbl, _ in val_dl:
            c5  = c5.to(DEVICE); cl= cl.to(DEVICE)
            oc  = oc.to(DEVICE); ar= ar.to(DEVICE)
            qv  = qv.to(DEVICE); qa= qa.to(DEVICE)
            qo  = qo.to(DEVICE); qs= qs.to(DEVICE)
            lbl = lbl.to(DEVICE)
            with autocast('cuda'):
                out = model(c5, cl, oc, ar, qv, qa, qo, qs, labels=lbl)
            val_loss += out.loss.item(); vsteps += 1

    avg_val = val_loss / vsteps
    scheduler.step()
    p1_history['train'].append(avg_trn)
    p1_history['val'].append(avg_val)

    print(f'[P1] Ep {epoch:02d}/{PHASE1_EPOCHS} | '
          f'train={avg_trn:.4f} | val={avg_val:.4f} | lr={scheduler.get_last_lr()[0]:.2e}')

    if avg_val < best_p1:
        best_p1 = avg_val
        torch.save({'epoch': epoch, 'phase': 1,
                    'model_state': model.state_dict(),
                    'val_loss': avg_val},
                   CKPT_DIR / 'phase1_best.pt')
        print(f'  Saved (val={avg_val:.4f})')

(CKPT_DIR / 'history_phase1.json').write_text(json.dumps(p1_history))
print(f'Phase 1 done. Best val CE: {best_p1:.4f}')
print('Target: < 4.0 to proceed to Phase 2')
torch.cuda.empty_cache()

## Cell 12 — Phase 2: Joint Fine-Tuning (10 epochs)

Unfreeze **everything** — fusion encoder + LoRA adapters — and fine-tune end-to-end.
Lower LR prevents destroying Phase 0 alignment.

> **Time:** ~25-30 min on T4

In [ ]:
import json

# Load best Phase 1 checkpoint
ckpt1 = torch.load(CKPT_DIR / 'phase1_best.pt', map_location=DEVICE)
model.load_state_dict(ckpt1['model_state'])
print(f'Loaded Phase 1 best (ep={ckpt1["epoch"]} val={ckpt1["val_loss"]:.4f})')

# Unfreeze fusion encoder
for p in model.encoder.parameters():
    p.requires_grad = True
trainable_p2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Phase 2 trainable: {trainable_p2/1e6:.2f}M  (fusion + LoRA)')

optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=PHASE2_EPOCHS, eta_min=1e-6)
scaler    = GradScaler('cuda')

p2_history = {'train': [], 'val': []}
best_p2    = ckpt1['val_loss']

print(f'Starting Phase 2 — {PHASE2_EPOCHS} epochs')

for epoch in range(1, PHASE2_EPOCHS + 1):

    model.train()
    trn_loss, steps = 0.0, 0
    optimizer.zero_grad()

    for step, (c5, cl, oc, ar, qv, qa, qo, qs, lbl, _) in enumerate(trn_dl):
        c5  = c5.to(DEVICE);  cl  = cl.to(DEVICE)
        oc  = oc.to(DEVICE);  ar  = ar.to(DEVICE)
        qv  = qv.to(DEVICE);  qa  = qa.to(DEVICE)
        qo  = qo.to(DEVICE);  qs  = qs.to(DEVICE)
        lbl = lbl.to(DEVICE)

        with autocast('cuda'):
            out  = model(c5, cl, oc, ar, qv, qa, qo, qs, labels=lbl)
            loss = out.loss / ACCUM_STEPS

        scaler.scale(loss).backward()
        if (step + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad()

        trn_loss += out.loss.item(); steps += 1

    avg_trn = trn_loss / steps

    model.eval()
    val_loss, vsteps = 0.0, 0
    with torch.no_grad():
        for c5, cl, oc, ar, qv, qa, qo, qs, lbl, _ in val_dl:
            c5  = c5.to(DEVICE); cl= cl.to(DEVICE)
            oc  = oc.to(DEVICE); ar= ar.to(DEVICE)
            qv  = qv.to(DEVICE); qa= qa.to(DEVICE)
            qo  = qo.to(DEVICE); qs= qs.to(DEVICE)
            lbl = lbl.to(DEVICE)
            with autocast('cuda'):
                out = model(c5, cl, oc, ar, qv, qa, qo, qs, labels=lbl)
            val_loss += out.loss.item(); vsteps += 1

    avg_val = val_loss / vsteps
    scheduler.step()
    p2_history['train'].append(avg_trn)
    p2_history['val'].append(avg_val)

    print(f'[P2] Ep {epoch:02d}/{PHASE2_EPOCHS} | '
          f'train={avg_trn:.4f} | val={avg_val:.4f} | lr={scheduler.get_last_lr()[0]:.2e}')

    if avg_val < best_p2:
        best_p2 = avg_val
        torch.save({'epoch': epoch, 'phase': 2,
                    'model_state': model.state_dict(),
                    'val_loss': avg_val},
                   CKPT_DIR / 'phase2_best.pt')
        print(f'  Saved (val={avg_val:.4f})')

(CKPT_DIR / 'history_phase2.json').write_text(json.dumps(p2_history))
print(f'Phase 2 done. Best val CE: {best_p2:.4f}')
torch.cuda.empty_cache()

## Cell 13 — Training History (All Phases)

In [ ]:
import json, matplotlib.pyplot as plt

h1 = json.loads((CKPT_DIR / 'history_phase1.json').read_text())
h2 = json.loads((CKPT_DIR / 'history_phase2.json').read_text())
ce_train = h1['train'] + h2['train']
ce_val   = h1['val']   + h2['val']
eps      = list(range(1, len(ce_train) + 1))
p1_len   = len(h1['train'])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(eps, ce_train, label='train CE loss', color='royalblue')
ax.plot(eps, ce_val,   label='val CE loss',   color='tomato')
ax.axvline(p1_len, color='gray', linestyle='--', alpha=0.7, label='P1 -> P2 (unfreeze fusion)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-entropy loss')
ax.set_title('PRISM-FUSION v4 — Phase 1 + Phase 2 Training')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CKPT_DIR / 'training_history.png', dpi=150)
plt.show()

## Cell 14 — Load Best Checkpoint

In [ ]:
# Check which phase gave better val loss
import os

p1_ckpt = CKPT_DIR / 'phase1_best.pt'
p2_ckpt = CKPT_DIR / 'phase2_best.pt'

best_ckpt = p2_ckpt if p2_ckpt.exists() else p1_ckpt
if p1_ckpt.exists() and p2_ckpt.exists():
    v1 = torch.load(p1_ckpt, map_location='cpu')['val_loss']
    v2 = torch.load(p2_ckpt, map_location='cpu')['val_loss']
    best_ckpt = p2_ckpt if v2 <= v1 else p1_ckpt
    print(f'Phase 1 best val: {v1:.4f} | Phase 2 best val: {v2:.4f}')
    print(f'Using: {best_ckpt.name}')

ckpt  = torch.load(best_ckpt, map_location=DEVICE)
model = PRISMFusionV4(encoder_ckpt=None).to(DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded | phase={ckpt["phase"]} | epoch={ckpt["epoch"]} | val={ckpt["val_loss"]:.4f}')

## Cell 15 — Inference on All Videos

In [ ]:
import pandas as pd, numpy as np

gt_full = pd.read_csv(GT_CSV, encoding='latin-1')
results = []

model.eval()
with torch.no_grad():
    for _, row in gt_full.iterrows():
        vid = row['video_id']
        c5_f  = CLIP5_DIR / f'{vid}.npy'
        cl_f  = CLAP_DIR  / f'{vid}.npy'
        oc_f  = BERT_DIR  / f'{vid}_ocr.npy'
        ar_f  = BERT_DIR  / f'{vid}_asr.npy'
        if not all(f.exists() for f in [c5_f, cl_f, oc_f, ar_f]):
            continue

        clap_np = np.load(cl_f)
        ocr_txt = load_text(OCR_DIR / f'{vid}.txt')
        asr_txt = load_text(ASR_DIR / f'{vid}.txt')

        c5 = torch.from_numpy(np.load(c5_f)).float().unsqueeze(0).to(DEVICE)
        cl = torch.from_numpy(clap_np).float().unsqueeze(0).to(DEVICE)
        oc = torch.from_numpy(np.load(oc_f)).float().unsqueeze(0).to(DEVICE)
        ar = torch.from_numpy(np.load(ar_f)).float().unsqueeze(0).to(DEVICE)

        qv = torch.tensor([1.0]).to(DEVICE)
        qa = torch.tensor([min(float(cl.norm().item()), 1.0)]).to(DEVICE)
        qo = torch.tensor([compute_quality(ocr_txt)]).to(DEVICE)
        qs = torch.tensor([compute_quality(asr_txt)]).to(DEVICE)

        pred = model.generate_title(c5, cl, oc, ar, qv, qa, qo, qs, bart_tokenizer)
        results.append({'video_id': vid, 'gt': row['title'], 'pred': pred,
                        'q_ocr': qo.item(), 'q_asr': qs.item(),
                        'q_audio': qa.item()})
        print(f'[{vid}]\n  GT  : {row["title"]}\n  PRED: {pred}\n')

df_res = pd.DataFrame(results)
df_res.to_csv(CKPT_DIR / 'predictions_v4.csv', index=False)
print(f'Saved {len(df_res)} predictions')

## Cell 16 — Evaluation

In [ ]:
import nltk, pandas as pd, numpy as np
from collections import Counter
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer as rouge_lib
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

df_ev  = pd.read_csv(CKPT_DIR / 'predictions_v4.csv')
preds  = df_ev['pred'].tolist()
gts    = df_ev['gt'].tolist()

scorer = rouge_lib.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
r1, r2, rl = [], [], []
for g, p in zip(gts, preds):
    s=scorer.score(g,p); r1.append(s['rouge1'].fmeasure)
    r2.append(s['rouge2'].fmeasure); rl.append(s['rougeL'].fmeasure)

refs  = [[g.lower().split()] for g in gts]
hyps  = [p.lower().split() for p in preds]
bleu4 = corpus_bleu(refs, hyps, smoothing_function=SmoothingFunction().method1)
bleu1 = corpus_bleu(refs, hyps, weights=(1,0,0,0),
                    smoothing_function=SmoothingFunction().method1)

def distinct_n(texts, n):
    ng=[]
    for t in texts:
        w=t.lower().split(); ng+=[tuple(w[i:i+n]) for i in range(len(w)-n+1)]
    return len(set(ng))/max(len(ng),1)

d3  = distinct_n(preds, 3)
rep = sum(Counter(p.lower().split()).most_common(1)[0][1]/max(len(p.split()),1) > 0.5
          for p in preds if len(p.split())>3)/len(preds)*100

# BERTScore
try:
    from bert_score import score as bscore
    import os; os.environ['TOKENIZERS_PARALLELISM']='false'
    _, _, F1 = bscore(preds, gts, lang='en',
                      model_type='bert-base-uncased', device=DEVICE, verbose=False)
    bs = F1.mean().item()
except Exception as e:
    print(f'BERTScore failed: {e}'); bs = float('nan')

# Per-modality breakdown
text_vids = df_ev[(df_ev.q_ocr>0.3) | (df_ev.q_asr>0.3)]
silent_vids = df_ev[(df_ev.q_ocr<=0.3) & (df_ev.q_asr<=0.3)]

rl_text   = sum(scorer.score(g,p)['rougeL'].fmeasure
               for g,p in zip(text_vids.gt, text_vids.pred))/max(len(text_vids),1)
rl_silent = sum(scorer.score(g,p)['rougeL'].fmeasure
               for g,p in zip(silent_vids.gt, silent_vids.pred))/max(len(silent_vids),1)

print('='*60)
print('PRISM-FUSION v4 — Evaluation Results')
print('='*60)
print(f'  Videos total        : {len(df_ev)}')
print(f'  Text-available      : {len(text_vids)}  ROUGE-L={rl_text:.4f}')
print(f'  Silent/visual-only  : {len(silent_vids)}  ROUGE-L={rl_silent:.4f}')
print()
if bs==bs: print(f'  BERTScore F1 : {bs:.4f}')
print(f'  ROUGE-1      : {sum(r1)/len(r1):.4f}')
print(f'  ROUGE-2      : {sum(r2)/len(r2):.4f}')
print(f'  ROUGE-L      : {sum(rl)/len(rl):.4f}')
print(f'  BLEU-1       : {bleu1:.4f}')
print(f'  BLEU-4       : {bleu4:.4f}')
print(f'  Distinct-3   : {d3:.4f}')
print(f'  Repetition % : {rep:.1f}%')
print()
print( '  Baselines:')
print( '    Random init multimodal (v1)  : BERTScore~0.60  ROUGE-L~0.05')
print( '    Text-only BART zero-shot     : BERTScore=0.836 ROUGE-L=0.133')
print('='*60)

pd.DataFrame([{'model':'prism_fusion_v4', 'n':len(df_ev),
               'BERTScore':bs, 'ROUGE1':sum(r1)/len(r1),
               'ROUGE2':sum(r2)/len(r2), 'ROUGE_L':sum(rl)/len(rl),
               'BLEU1':bleu1, 'BLEU4':bleu4, 'Distinct3':d3,
               'Rep_pct':rep}]).to_csv(CKPT_DIR/'metrics_v4.csv', index=False)
print('Metrics saved -> metrics_v4.csv')

In [ ]:
# ----- EXPORT FOR API DEPLOYMENT -----
import os

export_dir = CKPT_DIR / 'api_export_v4'
export_dir.mkdir(exist_ok=True, parents=True)

# 1. Save the LoRA adapters using PEFT
model.bart.save_pretrained(export_dir / 'lora_adapter')

# 2. Save the Fusion Encoder weights separately
torch.save(model.encoder.state_dict(), export_dir / 'fusion_encoder.pt')

print(f"Model components successfully exported to {export_dir}")
print("Download these files to deploy your API.")
